# 🛡️ EconoCausal: DoWhy Causal Audit & Refutation Suite
### Falsification Testing and Mathematical Validation of Treatment Effects

In predictive machine learning, validation is straightforward: you split data into train/test sets and measure loss or accuracy.

In **Causal AI**, counterfactuals cannot be directly observed (we cannot observe what a customer would have done both with and without a discount at the exact same moment). Therefore, proving that a model has uncovered true causality requires **Refutation Testing (Falsification Algorithms)**.

In this notebook, we use Microsoft's **DoWhy** library to stress-test our pricing model with three mathematical refutations:
1. **Placebo Treatment Refuter**: Replaces real discount assignments with random noise. If the estimated effect does not collapse to zero, the model was learning spurious correlation.
2. **Random Common Cause Refuter**: Injects an artificial independent confounder into the data. The treatment effect estimate should remain stable.
3. **Data Subset Refuter**: Retrains the estimator across random 80% slices of the data to verify that results are not driven by outlier subsets.


## 1. Imports and Environment Setup
We import DoWhy, NetworkX, and apply compatibility patches ensuring smooth operation on modern NetworkX 3.x and Pandas 2.x environments.

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. NetworkX compatibility patch for DoWhy on NetworkX 3.x
import networkx as nx
import networkx.algorithms.d_separation as ds
if not hasattr(nx.algorithms, 'd_separated'):
    nx.algorithms.d_separated = ds.is_d_separator

# 2. Pandas 2.x compatibility patch for RegressionEstimator in DoWhy 0.8
import dowhy.causal_estimators.regression_estimator as re_module
def _patched_estimate_effect(self, data_df=None, need_conditional_estimates=None):
    if data_df is None:
        data_df = self._data
    if need_conditional_estimates is None:
        need_conditional_estimates = self.need_conditional_estimates
    if not self.model:
        _, self.model = self._build_model()
    effect_estimate = self._do(self._treatment_value, data_df) - self._do(self._control_value, data_df)
    conditional_effect_estimates = None
    if need_conditional_estimates:
        conditional_effect_estimates = self._estimate_conditional_effects(
            self._estimate_effect_fn,
            effect_modifier_names=self._effect_modifier_names)
    intercept_parameter = self.model.params.iloc[0] if hasattr(self.model.params, 'iloc') else self.model.params[0]
    estimate = re_module.CausalEstimate(
        estimate=effect_estimate,
        control_value=self._control_value,
        treatment_value=self._treatment_value,
        conditional_estimates=conditional_effect_estimates,
        target_estimand=self._target_estimand,
        realized_estimand_expr=self.symbolic_estimator,
        intercept=intercept_parameter
    )
    return estimate

re_module.RegressionEstimator._estimate_effect = _patched_estimate_effect

from dowhy import CausalModel
from features import build_feature_pipeline

# Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)

print("Environment setup and compatibility patches successfully loaded!")


Environment setup and compatibility patches successfully loaded!


## 2. Ingesting Data & Covariates
We extract 15,000 records from the feature pipeline, focusing on the binary treatment contrast: **$20 Discount vs. $0 Control**.

In [2]:
# Ingest dataset using feature pipeline
data = build_feature_pipeline(sample_size=15000, random_state=42)
df = data['covariates'].copy()
df['target'] = data['Y'].values

# Filter for binary treatment: $20 vs $0
binary_mask = data['T'].isin([0, 20])
df_binary = df[binary_mask].copy()
df_binary['treatment_20'] = (data['T'][binary_mask] == 20).astype(int)

print(f"Total audit samples: {len(df_binary):,}")
print(f"Treated ($20 discount): {df_binary['treatment_20'].sum():,}")
print(f"Control ($0 discount):  {(df_binary['treatment_20'] == 0).sum():,}")
print(f"Base purchase rate:     {df_binary['target'].mean()*100:.2f}%")

df_binary.head()


Loading data from data/synthetic_dynamic_pricing.csv and data/clients.csv...


Subsampling to 15000 records for training/iteration...
Engineering demographic and behavioral features...
Feature pipeline complete! Processed 15000 clients.
Treatment distribution:
discount_amount
0     7491
10    3701
20    3808
Name: count, dtype: int64
Target conversion rate: 0.6267
Total audit samples: 11,299
Treated ($20 discount): 3,808
Control ($0 discount):  7,491
Base purchase rate:     62.36%


,age,gender_M,gender_F,gender_U,has_redeemed,first_issue_days,first_redeem_days,issue_redeem_delay,target,treatment_20
client_id,,,,,,,,,,
ac126a8be6,27,0,1,0,1,172,887.0,715.0,0,0
ba4a67aba7,81,0,1,0,0,358,0.0,0.0,1,0
b2fc4ea450,18,0,1,0,1,429,996.0,567.0,0,0
67dc2d5e46,31,1,0,0,1,169,417.0,248.0,0,0
7b061de2b1,57,0,1,0,1,192,315.0,123.0,1,0
